# Task 4: SAM Classification and Sensor Calibration
This notebook implements the Spectral Angle Mapper (SAM) for airborne data classification and performs a radiometric calibration of Sentinel-2 data using the airborne data as a reference.

## Step 1: Imports and Configuration
We load libraries for hyperspectral processing, STAC discovery, and statistical analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform
from rasterio.windows import from_bounds
import spectral.io.envi as envi
from pystac_client import Client
import planetary_computer
from datetime import datetime
from scipy.stats import linregress
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

ACQUISITION_DATE = '2025-06-17'
BASE_DIR = Path().cwd() / "data" / "you-shall-not-pass" / "Obrazy lotnicze"
hdr_path = BASE_DIR / '221000_Odra_HS_Blok_A_008_VS_join_atm.hdr'

## Description of Methods

1. **Spectral Angle Mapper (SAM)**: A physically-based spectral classification that uses an n-D angle to match pixels to reference spectra. It is insensitive to illumination differences (shadows vs bright areas) because it compares the 'shape' of the spectrum rather than its magnitude.
2. **Linear Regression Calibration**: We match Sentinel-2 reflectance to airborne reflectance by finding a linear relationship ($y = mx + c$). This accounts for different sensor sensitivities and atmospheric effects.
3. **Spatial Aggregation**: Since airborne data is 1m and Sentinel-2 is 10m, we aggregate 10x10 airborne pixels into one to create a comparable dataset for the calibration regression.
4. **Geographic Re-alignment**: We use the airborne coordinate system metadata to clip the exact matching pixels from the global Sentinel-2 tile.

## Step 2: Load Airborne Data
We open the cube and extract the geographic bounds to ensure perfect alignment with satellite data.

In [ ]:
img = envi.open(hdr_path)
meta = img.metadata
wavelengths = np.array([float(x) for x in meta['wavelength']])

# Subset 1000x1000 from center
cy, cx = img.nrows // 2, img.ncols // 2
airborne_cube = img.read_subregion((cy-500, cy+500), (cx-500, cx+500))

with rasterio.open(hdr_path) as src:
    air_win = rasterio.windows.Window(cx-500, cy-500, 1000, 1000)
    air_left, air_bottom, air_right, air_top = src.window_bounds(air_win)
    lons, lats = transform(src.crs, 'EPSG:4326', [air_left, air_right], [air_bottom, air_top])
    subset_bbox = [lons[0], lats[0], lons[1], lats[1]]

## Step 3: SAM Classification
We classify the image based on the spectral libraries for water, forest, and green areas.

In [ ]:
sig_dir = Path('spectral_signatures')
spectra = {}
for f in sig_dir.glob('*.csv'):
    class_name = f.stem.rstrip('1234567890')
    df = pd.read_csv(f)
    if class_name not in spectra: spectra[class_name] = []
    spectra[class_name].append(df['value'].values)

reference_spectra = {k: np.nanmean(v, axis=0) for k, v in spectra.items()}

def compute_sam(cube, ref):
    if len(ref) != cube.shape[2]:
        ref = np.interp(np.linspace(0, 1, cube.shape[2]), np.linspace(0, 1, len(ref)), ref)
    dot = np.sum(cube * ref, axis=2)
    norm_c = np.sqrt(np.sum(cube**2, axis=2))
    norm_r = np.sqrt(np.sum(ref**2))
    return np.arccos(np.clip(dot / (norm_c * norm_r + 1e-8), -1, 1))

sam_maps = {k: compute_sam(airborne_cube, ref) for k, ref in reference_spectra.items()}
classification = np.argmin(np.stack(list(sam_maps.values()), axis=-1), axis=-1)

plt.figure(figsize=(8, 6))
plt.imshow(classification, cmap='Set1')
plt.title("SAM Classification Map (Airborne)")
plt.show()

## Step 4: Fetch Aligned Sentinel-2 Data
We use the STAC API to find the exact matching scene for the airborne acquisition date.

In [ ]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)
search = catalog.search(collections=["sentinel-2-l2a"], bbox=subset_bbox, datetime=ACQUISITION_DATE, query={"eo:cloud_cover": {"lt": 10}})
items = list(search.items())
best_item = items[0]

with rasterio.open(best_item.assets['B04'].href) as src:
    s2_win = from_bounds(subset_bbox[0], subset_bbox[1], subset_bbox[2], subset_bbox[3], src.transform)
    s2_red = src.read(1, window=s2_win).astype(float) / 10000.0

## Step 5: Radiometric Calibration
We compute the regression between S2 and Airborne data. We use a histogram to visualize the shift.

In [ ]:
# 1. Downsample airborne to 10m
red_idx = np.argmin(np.abs(wavelengths - 665))
ab_red = airborne_cube[:, :, red_idx]
h, w = ab_red.shape
ab_10m = ab_red[:h//10*10, :w//10*10].reshape(h//10, 10, w//10, 10).mean(axis=(1, 3))

# 2. Match sizes
s2_sub = s2_red[:ab_10m.shape[0], :ab_10m.shape[1]]
mask = (ab_10m > 0) & (s2_sub > 0)

# 3. Regression
res = linregress(s2_sub[mask], ab_10m[mask])
s2_calibrated = s2_red * res.slope + res.intercept

print(f"Slope: {res.slope:.4f}, Intercept: {res.intercept:.4f}, R2: {res.rvalue**2:.4f}")

plt.figure(figsize=(12, 5))
plt.hist(s2_red.flatten(), bins=50, alpha=0.5, label='Original Sentinel-2')
plt.hist(s2_calibrated.flatten(), bins=50, alpha=0.5, label='Calibrated (Matched to Airborne)')
plt.title("Evidence of Calibration: Histogram Shift")
plt.legend()
plt.show()

## Step 6: Visualizing the Difference
By plotting the difference map, we can see exactly how much the calibration adjusted each pixel.

In [ ]:
plt.figure(figsize=(8, 6))
diff = s2_calibrated - s2_red
im = plt.imshow(diff, cmap='RdBu_r', vmin=-0.05, vmax=0.05)
plt.colorbar(im, label="Reflectance Change")
plt.title("Calibration Difference Map (Calibrated - Original)")
plt.show()